# Pokédle — Análisis Exploratorio de Datos (EDA)

Este notebook demuestra habilidades de Data Engineering y Análisis de Datos:
- Lectura desde una base de datos SQLite (no solo CSV)
- Limpieza y validación de datos con pandas
- Visualizaciones con matplotlib y seaborn
- Análisis de estadísticas de juego
- Identificación de pokémones más difíciles de adivinar


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DB_PATH = os.path.join('..', 'data', 'pokedle.db')
conn = sqlite3.connect(DB_PATH)
print('Conexión exitosa a', DB_PATH)

## 1. Carga y validación de datos

In [ ]:
df = pd.read_sql('SELECT * FROM pokemon ORDER BY id', conn)

print(f'Filas: {len(df)}')
print(f'Columnas: {list(df.columns)}')
print(f'\nTipos de datos:')
print(df.dtypes)

df.head()

In [ ]:
# Validación de calidad de datos
print('=== Validación de calidad ===')
print(f'Valores nulos por columna:')
print(df.isnull().sum())
print(f'\nDuplicados: {df.duplicated().sum()}')
print(f'Rango de IDs: {df.id.min()} – {df.id.max()}')
print(f'Etapas de evolución únicas: {sorted(df.etapa_evolucion.unique())}')
print(f'Generaciones únicas: {sorted(df.generacion.unique())}')

## 2. Distribución por tipos

In [ ]:
# Contar pokémones por tipo primario
conteo_tipo1 = df['tipo_1'].value_counts().reset_index()
conteo_tipo1.columns = ['tipo', 'cantidad']

# Paleta de colores por tipo (colores canónicos de Pokémon)
colores_tipo = {
    'normal': '#A8A878', 'fire': '#F08030', 'water': '#6890F0',
    'electric': '#F8D030', 'grass': '#78C850', 'ice': '#98D8D8',
    'fighting': '#C03028', 'poison': '#A040A0', 'ground': '#E0C068',
    'flying': '#A890F0', 'psychic': '#F85888', 'bug': '#A8B820',
    'rock': '#B8A038', 'ghost': '#705898', 'dragon': '#7038F8',
    'dark': '#705848', 'steel': '#B8B8D0', 'fairy': '#EE99AC',
}

fig, ax = plt.subplots(figsize=(14, 6))
barras = ax.bar(
    conteo_tipo1['tipo'],
    conteo_tipo1['cantidad'],
    color=[colores_tipo.get(t, '#888') for t in conteo_tipo1['tipo']],
    edgecolor='white', linewidth=0.8
)
ax.set_title('Distribución de pokémones por Tipo Primario (Gen 1)', fontsize=16, fontweight='bold')
ax.set_xlabel('Tipo', fontsize=13)
ax.set_ylabel('Cantidad de pokémones', fontsize=13)
plt.xticks(rotation=45, ha='right')

# Etiquetas encima de cada barra
for barra in barras:
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.1,
            int(barra.get_height()), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('distribucion_tipos.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Tipo más común: {conteo_tipo1.iloc[0].tipo} ({conteo_tipo1.iloc[0].cantidad} pokémones)')
print(f'Tipo menos común: {conteo_tipo1.iloc[-1].tipo} ({conteo_tipo1.iloc[-1].cantidad} pokémones)')

## 3. Estadísticas descriptivas — Altura y Peso

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histograma de altura
axes[0].hist(df['altura_m'], bins=20, color='#6890F0', edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribución de Altura (m)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Altura (metros)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df['altura_m'].mean(), color='red', linestyle='--', label=f'Media: {df.altura_m.mean():.2f}m')
axes[0].axvline(df['altura_m'].median(), color='orange', linestyle='--', label=f'Mediana: {df.altura_m.median():.2f}m')
axes[0].legend()

# Histograma de peso
axes[1].hist(df['peso_kg'], bins=25, color='#78C850', edgecolor='white', linewidth=0.8)
axes[1].set_title('Distribución de Peso (kg)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Peso (kg)')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(df['peso_kg'].mean(), color='red', linestyle='--', label=f'Media: {df.peso_kg.mean():.1f}kg')
axes[1].axvline(df['peso_kg'].median(), color='orange', linestyle='--', label=f'Mediana: {df.peso_kg.median():.1f}kg')
axes[1].legend()

plt.suptitle('Distribuciones físicas — Generación 1', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('distribuciones_fisicas.png', dpi=150, bbox_inches='tight')
plt.show()

print(df[['altura_m', 'peso_kg']].describe().round(2))
print(f'\nPokémon más alto: {df.loc[df.altura_m.idxmax(), "nombre"]} ({df.altura_m.max()}m)')
print(f'Pokémon más pesado: {df.loc[df.peso_kg.idxmax(), "nombre"]} ({df.peso_kg.max()}kg)')

## 4. Correlación entre atributos (heatmap)

In [ ]:
numericas = ['altura_m', 'peso_kg', 'etapa_evolucion', 'generacion']
corr = df[numericas].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5,
    ax=ax
)
ax.set_title('Correlación entre atributos numéricos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlacion_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlación altura–peso:', round(corr.loc['altura_m','peso_kg'], 3))
print('Correlación altura–etapa_evolucion:', round(corr.loc['altura_m','etapa_evolucion'], 3))

## 5. Stats base (join con tabla stats)

In [ ]:
# Join de pokemon + stats
df_stats = pd.read_sql('SELECT * FROM stats', conn)
df_stats_pivot = df_stats.pivot(index='id_pokemon', columns='stat', values='valor').reset_index()
df_full = df.merge(df_stats_pivot, left_on='id', right_on='id_pokemon', how='left')

stats_cols = ['hp', 'attack', 'defense', 'special-attack', 'special-defense', 'speed']
stats_cols = [c for c in stats_cols if c in df_full.columns]

print('Stats disponibles:', stats_cols)
print(df_full[stats_cols].describe().round(1))

In [ ]:
# Radar chart del promedio de stats por tipo
stats_por_tipo = df_full.groupby('tipo_1')[stats_cols].mean().round(1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, stat in enumerate(stats_cols):
    data_sorted = stats_por_tipo[stat].sort_values(ascending=False)
    colores = [colores_tipo.get(t, '#888') for t in data_sorted.index]
    axes[i].barh(data_sorted.index, data_sorted.values, color=colores)
    axes[i].set_title(f'Media de {stat}', fontweight='bold')
    axes[i].set_xlabel('Puntos base')

plt.suptitle('Estadísticas base promedio por tipo primario', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('stats_por_tipo.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Análisis de partidas jugadas

In [ ]:
df_partidas = pd.read_sql('SELECT * FROM partidas', conn)
df_intentos = pd.read_sql('SELECT * FROM intentos', conn)

if df_partidas.empty:
    print('Aún no hay partidas registradas. Juega algunas partidas primero.')
else:
    print(f'Partidas jugadas: {len(df_partidas)}')
    print(f'Ganadas: {(df_partidas.ganada == 1).sum()} ({100*(df_partidas.ganada==1).mean():.0f}%)')
    print(f'Perdidas: {(df_partidas.ganada == 2).sum()}')
    print(f'Promedio de intentos (en partidas ganadas): {df_partidas[df_partidas.ganada==1].num_intentos.mean():.1f}')

    # Distribución de intentos hasta ganar
    ganadas = df_partidas[df_partidas.ganada == 1]
    if len(ganadas) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(ganadas['num_intentos'], bins=range(1, 10), color='#78C850', edgecolor='white', align='left')
        ax.set_title('Distribución de intentos para ganar', fontsize=14, fontweight='bold')
        ax.set_xlabel('Número de intentos')
        ax.set_ylabel('Partidas')
        ax.set_xticks(range(1, 9))
        plt.tight_layout()
        plt.savefig('distribucion_intentos.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# Pokémones más difíciles (más intentos promedio cuando son el objetivo)
if not df_partidas.empty and not df_intentos.empty:
    dificultad = (
        df_partidas[df_partidas.ganada == 1]
        .merge(df[['id', 'nombre']], left_on='id_pokemon_obj', right_on='id')
        .groupby('nombre')['num_intentos']
        .agg(['mean', 'count'])
        .query('count >= 2')
        .sort_values('mean', ascending=False)
        .head(10)
    )
    
    if not dificultad.empty:
        print('Pokémones más difíciles de adivinar:')
        print(dificultad.round(2))
    else:
        print('Necesitas más partidas para calcular dificultad.')

conn.close()
print('\nAnálisis completo.')